In [140]:
import numpy as np
import pandas as pd
import re


In [161]:
df = pd.read_csv('dataWithCarModel.csv')

In [162]:
# how many rows don't have brand
print("The rows that don't have brand are:", df['brand'].isna().sum())

# how many rows don't have model
print("The rows that don't have model are:", df['model'].isna().sum())

# how many rows have no brand and model
print("The rows that don't have brand AND model are:", df[df['brand'].isna() & df['model'].isna()].shape[0])


The rows that don't have brand are: 25257
The rows that don't have model are: 25257
The rows that don't have brand AND model are: 25257


In [163]:
count = df[df['brand'].str.lower().isin(['mercedes', 'benz'])].shape[0]
print("There are", count, "rows with brand 'mercedes' or 'benz'.")
df['brand'] = df['brand'].str.lower().replace({
    'mercedes': 'mercedes-benz',
    'benz': 'mercedes-benz'
})

There are 32 rows with brand 'mercedes' or 'benz'.


FROM THIS OUTPUT WE UNDERSTAND THAT IF THE BRAND IS MISSING ALSO THE MODEL IS MISSING AND VICE VERSA.
NOW I'M GOING TO TAKE ALL THE BRANDS AND MODELS THAT I FOUND OUT SO FAR, AND TRY TO FIND OUT IF I CAN COMPLETE THE OTHER COLUMNS WHERE THERE IS NO BRAND AN MODEL

In [164]:
# NOW i NEED TO TAKE ALL THE PAIRS BRAND AND MODEL

brand_model_pairs = (
    df.dropna(subset=['brand', 'model'])[['brand', 'model']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Count how many brands are associated with each model
model_brand_counts = brand_model_pairs.groupby('model')['brand'].nunique()
# Filter models that appear under more than one brand
shared_models = model_brand_counts[model_brand_counts >= 1]

print("Models used by multiple brands:\n")
print(shared_models.sort_values(ascending=False))
# there are no brands with the same model of the car


Models used by multiple brands:

model
#1            1
1007          1
107           1
108           1
124 spider    1
             ..
zoe           1
zr            1
zr-v          1
zs            1
zs ev         1
Name: brand, Length: 692, dtype: int64


In [165]:
def fill(df, brand_model_pairs):
    df1 = df.copy()

    # Normalize text for consistent comparison
    df1['brand_model'] = df1['brand_model'].astype(str).str.lower()
    brand_model_pairs['model'] = brand_model_pairs['model'].astype(str).str.strip().str.lower()
    brand_model_pairs['brand'] = brand_model_pairs['brand'].astype(str).str.strip().str.title()

    # Step 1: Create lookup dictionary: model -> brand
    model_to_brand = dict(zip(brand_model_pairs['model'], brand_model_pairs['brand']))

    keys = model_to_brand.keys()
    
    # ok we got the models now
    # take the rows where the model or the brand or both are equal to ""
    mask = (
        df1['brand'].fillna('').str.strip() == ''
    ) | (
        df1['model'].fillna('').str.strip() == ''
    )

    df_missing = df1[mask]

    # for every row in df_missing and for every row in keys
    # if key is inside brand_model, then update brand and model
    for idx, row in df_missing.iterrows():
        title = row['brand_model']
        for model in keys:
            if model in title:
                df1.at[idx, 'model'] = model.title()
                df1.at[idx, 'brand'] = model_to_brand[model]
                break  # stop at first match

    return df1
df1 = fill(df, brand_model_pairs)


In [166]:
# how many rows don't have brand
print("The rows that don't have brand are:", df1['brand'].isna().sum())

# how many rows don't have model
print("The rows that don't have model are:", df1['model'].isna().sum())

# how many rows have no brand and model
print("The rows that don't have brand AND model are:", df1[df1['brand'].isna() & df1['model'].isna()].shape[0])

The rows that don't have brand are: 519
The rows that don't have model are: 519
The rows that don't have brand AND model are: 519


In [167]:
df =  df1[~(df1['brand'].isna() & df1['model'].isna())]
df['brand'] = df['brand'].astype(str).str.strip().replace('', np.nan)
df['model'] = df['model'].astype(str).str.strip().replace('', np.nan)

C:\Users\Enrico\AppData\Local\Temp\ipykernel_17880\2930950859.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['brand'] = df['brand'].astype(str).str.strip().replace('', np.nan)
C:\Users\Enrico\AppData\Local\Temp\ipykernel_17880\2930950859.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['model'] = df['model'].astype(str).str.strip().replace('', np.nan)


In [168]:
df['brand_model'] = df['brand_model'].astype(str).str.lower()
df['brand'] = df['brand'].str.lower()
df['model'] = df['model'].str.lower()

C:\Users\Enrico\AppData\Local\Temp\ipykernel_17880\3468187463.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['brand_model'] = df['brand_model'].astype(str).str.lower()
C:\Users\Enrico\AppData\Local\Temp\ipykernel_17880\3468187463.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['brand'] = df['brand'].str.lower()
C:\Users\Enrico\AppData\Local\Temp\ipykernel_17880\3468187463.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_index

In [169]:
brand_counts = df['brand'].value_counts()
brand_counts.to_csv("brand_counts.csv", header=["count"])

In [171]:
df.to_csv("dataCars.csv", index=False)